# NemoNexus - OLYMPUS Ensemble for ARC Prize 2025

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from typing import Dict, List, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
else:
    print('No GPU available - using CPU')

In [ ]:
class SpecialistModel(nn.Module):
    def __init__(self, max_grid_size=30, d_model=256, specialist_type='MINERVA'):
        super().__init__()
        self.specialist_type = specialist_type
        self.max_grid_size = max_grid_size
        self.d_model = d_model
        
        if specialist_type == 'MINERVA':
            self.num_layers = 6
            self.num_heads = 8
            self.d_model = 1024
        elif specialist_type == 'ATLAS':
            self.num_layers = 4
            self.num_heads = 8
            self.d_model = 512
        elif specialist_type == 'IRIS':
            self.num_layers = 3
            self.num_heads = 4
            self.d_model = 512
        elif specialist_type == 'CHRONOS':
            self.num_layers = 8
            self.num_heads = 8
            self.d_model = 1024
        elif specialist_type == 'PROMETHEUS':
            self.num_layers = 6
            self.num_heads = 6
            self.d_model = 768
        
        self.input_embedding = nn.Embedding(10, self.d_model)
        self.position_embedding = nn.Parameter(torch.randn(max_grid_size * max_grid_size, self.d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.num_heads,
            dim_feedforward=self.d_model * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers)
        self.output_projection = nn.Linear(self.d_model, 10)
        
    def forward(self, x):
        B, H, W = x.shape
        x_flat = x.view(B, -1)
        embedded = self.input_embedding(x_flat)
        seq_len = embedded.shape[1]
        embedded += self.position_embedding[:seq_len].unsqueeze(0)
        output = self.transformer(embedded)
        logits = self.output_projection(output)
        return logits.view(B, H, W, 10)

In [ ]:
class TaskRouter(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )
        
    def forward(self, input_features):
        weights = self.feature_extractor(input_features)
        return F.softmax(weights, dim=-1)

In [ ]:
class OLYMPUSEnsemble(nn.Module):
    def __init__(self, max_grid_size=30, d_model=256):
        super().__init__()
        self.max_grid_size = max_grid_size
        
        self.specialists = nn.ModuleDict({
            'MINERVA': SpecialistModel(max_grid_size, d_model, 'MINERVA'),
            'ATLAS': SpecialistModel(max_grid_size, d_model, 'ATLAS'),
            'IRIS': SpecialistModel(max_grid_size, d_model, 'IRIS'),
            'CHRONOS': SpecialistModel(max_grid_size, d_model, 'CHRONOS'),
            'PROMETHEUS': SpecialistModel(max_grid_size, d_model, 'PROMETHEUS')
        })
        
        self.task_router = TaskRouter(d_model)
        
    def forward(self, input_grid, train_examples=None, target_shape=None):
        B, H, W = input_grid.shape
        
        specialist_outputs = {}
        for name, specialist in self.specialists.items():
            with torch.no_grad():
                output = specialist(input_grid)
                specialist_outputs[name] = output
        
        ensemble_output = torch.stack(list(specialist_outputs.values())).mean(dim=0)
        return ensemble_output
    
    def load_trained_weights(self, model_path='/kaggle/input/nemonexus/pytorch/default/1/NemoNexus.pt'):
        try:
            if os.path.exists(model_path):
                checkpoint = torch.load(model_path, map_location=device)
                self.load_state_dict(checkpoint, strict=False)
                print(f'Loaded NemoNexus trained weights from {model_path}')
                return True
        except Exception as e:
            print(f'Could not load NemoNexus weights: {e}')
        
        print('Using randomly initialized weights')
        return False

In [ ]:
class NemoNexus:
    def __init__(self, model_dir=None):
        self.olympus = OLYMPUSEnsemble().to(device)
        
        if model_dir:
            model_path = os.path.join(model_dir, 'NemoNexus.pt')
            self.olympus.load_trained_weights(model_path)
        
    def predict(self, task_data):
        predictions = []
        train_examples = task_data['train']
        test_inputs = task_data['test']
        
        self.olympus.eval()
        
        for test_input in test_inputs:
            input_grid = np.array(test_input['input'])
            input_tensor = torch.tensor(input_grid, dtype=torch.long).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output_logits = self.olympus(input_tensor, train_examples)
                output_pred = torch.argmax(output_logits, dim=-1)
                pred_grid = output_pred[0].cpu().numpy()
            
            predictions.append({
                "attempt_1": pred_grid.tolist(),
                "attempt_2": pred_grid.tolist()
            })
        
        return predictions

In [ ]:
nemo = NemoNexus(model_dir='/kaggle/input/nemonexus/pytorch/default/1')
print('NemoNexus OLYMPUS ensemble initialized')

In [ ]:
with open('/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json', 'r') as f:
    test_challenges = json.load(f)

print(f'Loaded {len(test_challenges)} test challenges')

In [ ]:
solutions = {}
processed = 0

for task_id, task_data in test_challenges.items():
    try:
        predictions = nemo.predict(task_data)
        solutions[task_id] = predictions
        processed += 1
        
        if processed % 50 == 0:
            print(f'NemoNexus processed {processed}/{len(test_challenges)} tasks')
            
    except Exception as e:
        print(f'Error on task {task_id}: {e}')
        solutions[task_id] = [{"attempt_1": [[0]], "attempt_2": [[0]]}]

print(f'NemoNexus generated solutions for {len(solutions)} tasks')

In [ ]:
try:
    with open('submission.json', 'w') as f:
        json.dump(solutions, f, separators=(',', ':'))
    print('NemoNexus submission saved!')
    print(f'Total tasks: {len(solutions)}')
    print('Submission file: submission.json')
    
except Exception as e:
    print(f'Error saving submission: {e}')